# 🤖 Cellula — Intelligent AI Coding Assistant

This notebook walks through the complete implementation of an AI Coding Assistant with:
- **Intent Classification** (Explain vs Generate)
- **Code Explanation** (Direct LLM, no RAG)
- **Code Generation** with RAG (ChromaDB + Sentence Transformers)
- **Relevance Checking** for retrieved documents
- **Human Feedback Learning** to expand the knowledge base
- **Conversation Memory** for context-aware interactions
- **Code Execution** tool
- **Streamlit UI** integration

## 1. Install Dependencies

In [ ]:
!pip install streamlit huggingface-hub sentence-transformers chromadb langchain langchain-community torch transformers -q

## 2. Imports & Configuration

In [ ]:
import os
import re
import uuid
import tempfile
import subprocess
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from huggingface_hub import InferenceClient

# Set your Hugging Face token here
HF_TOKEN = "YOUR_HF_TOKEN_HERE"  # Replace with your token from https://huggingface.co/settings/tokens
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

## 3. Vector Store — ChromaDB Wrapper

This module handles document storage, embedding generation, and similarity search using ChromaDB with sentence-transformers embeddings.

In [ ]:
class VectorStore:
    """ChromaDB wrapper for document storage and retrieval."""
    
    def __init__(self, persist_directory='./chroma_db', collection_name='coding_knowledge'):
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.embedding_fn = SentenceTransformerEmbeddingFunction(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_fn
        )

    def add_document(self, content: str, metadata: dict = None) -> str:
        """Add a document to the vector store with automatic chunking."""
        chunks = [chunk.strip() for chunk in content.split('\n\n') if chunk.strip()]
        
        final_chunks = []
        for chunk in chunks:
            if len(chunk) > 500:
                words = chunk.split()
                current_chunk = []
                current_len = 0
                for word in words:
                    if current_len + len(word) + 1 > 500:
                        final_chunks.append(" ".join(current_chunk))
                        current_chunk = [word]
                        current_len = len(word)
                    else:
                        current_chunk.append(word)
                        current_len += len(word) + 1
                if current_chunk:
                    final_chunks.append(" ".join(current_chunk))
            else:
                final_chunks.append(chunk)

        doc_ids = []
        for chunk in final_chunks:
            doc_id = str(uuid.uuid4())
            self.collection.add(
                documents=[chunk],
                metadatas=[metadata or {}],
                ids=[doc_id]
            )
            doc_ids.append(doc_id)
        
        return doc_ids[0] if doc_ids else None

    def search(self, query: str, top_k: int = 3) -> list:
        """Search for similar documents."""
        if self.get_collection_count() == 0:
            return []
            
        results = self.collection.query(
            query_texts=[query],
            n_results=min(top_k, self.get_collection_count())
        )
        
        formatted_results = []
        if results['documents'] and len(results['documents']) > 0:
            for i in range(len(results['documents'][0])):
                formatted_results.append({
                    'content': results['documents'][0][i],
                    'metadata': results['metadatas'][0][i] if results['metadatas'] else {},
                    'distance': results['distances'][0][i] if 'distances' in results and results['distances'] else 0.0
                })
        return formatted_results

    def get_collection_count(self) -> int:
        """Return the number of documents in the collection."""
        return self.collection.count()

# Test Vector Store
vs = VectorStore()
print(f"Vector store initialized. Document count: {vs.get_collection_count()}")

## 4. Intent Classifier

Classifies user queries as either **EXPLAIN** (code explanation) or **GENERATE** (code generation). This determines which pipeline processes the request.

In [ ]:
class IntentClassifier:
    """LLM-based intent classifier for routing queries."""
    
    def __init__(self, client: InferenceClient, model_name: str):
        self.client = client
        self.model_name = model_name

    def classify(self, query: str) -> str:
        messages = [
            {'role': 'system', 'content': 'You are an intent classifier. Respond with ONLY the word "EXPLAIN" if the user wants code explained, or "GENERATE" if the user wants code generated/written. Do not add any other text.'},
            {'role': 'user', 'content': query}
        ]
        try:
            response = self.client.chat_completion(
                model=self.model_name,
                messages=messages,
                max_tokens=10
            )
            result = response.choices[0].message.content.strip().upper()
            if 'EXPLAIN' in result:
                return 'EXPLAIN'
            return 'GENERATE'
        except Exception as e:
            print(f"Classification error: {e}")
            return 'GENERATE'

# Test Intent Classifier
client = InferenceClient(token=HF_TOKEN)
classifier = IntentClassifier(client, MODEL_NAME)

test_queries = [
    "Explain this Python function",
    "Write a sorting algorithm",
    "What does this code do?",
    "Generate a Flask API"
]
for q in test_queries:
    print(f"Query: '{q}' → Intent: {classifier.classify(q)}")

## 5. Code Explainer

**Route 1 — Code Explanation**: This pipeline directly uses the LLM to explain code. It does NOT use RAG, no document retrieval, no vector database search, no embeddings.

In [ ]:
class CodeExplainer:
    """Direct LLM code explanation — no RAG involved."""
    
    def __init__(self, client: InferenceClient, model_name: str):
        self.client = client
        self.model_name = model_name

    def explain(self, code: str, memory_context: str = '') -> str:
        messages = [
            {'role': 'system', 'content': 'You are an expert Python developer. Explain the provided code clearly with line-by-line analysis where appropriate.'},
            {'role': 'user', 'content': f'Context:\n{memory_context}\n\nCode to explain:\n{code}'}
        ]
        response = self.client.chat_completion(
            model=self.model_name,
            messages=messages,
            max_tokens=1024
        )
        return response.choices[0].message.content

# Test Code Explainer
explainer = CodeExplainer(client, MODEL_NAME)
sample_code = '''def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)'''
print(explainer.explain(sample_code))

## 6. Relevance Checker

After retrieving documents from ChromaDB, this LLM evaluator judges whether the retrieved context is relevant to the user's query. This prevents hallucination from irrelevant context.

In [ ]:
class RelevanceChecker:
    """LLM-based relevance evaluator for retrieved documents."""
    
    def __init__(self, client: InferenceClient, model_name: str):
        self.client = client
        self.model_name = model_name

    def check(self, query: str, retrieved_context: str) -> bool:
        messages = [
            {'role': 'system', 'content': 'You are a relevance checker. Given a query and some context, respond with ONLY "TRUE" if the context contains information relevant to answering the query, or "FALSE" if it does not.'},
            {'role': 'user', 'content': f'Query: {query}\n\nContext:\n{retrieved_context}'}
        ]
        try:
            response = self.client.chat_completion(
                model=self.model_name,
                messages=messages,
                max_tokens=10
            )
            result = response.choices[0].message.content.strip().upper()
            return 'TRUE' in result
        except Exception:
            return False

# Test Relevance Checker
checker = RelevanceChecker(client, MODEL_NAME)
print("Relevant:", checker.check("How to sort a list?", "Python's sorted() function returns a new sorted list."))
print("Not relevant:", checker.check("How to sort a list?", "The weather today is sunny."))

## 7. RAG Code Generator

**Route 2 — Code Generation**: The full RAG pipeline:
1. Search ChromaDB for relevant documents
2. Check relevance of retrieved context
3. If relevant → Generate code using context + query
4. If not relevant → Ask user for feedback/solution

In [ ]:
class RAGCodeGenerator:
    """RAG-powered code generation with relevance checking."""
    
    def __init__(self, client: InferenceClient, vector_store: VectorStore, 
                 relevance_checker: RelevanceChecker, model_name: str):
        self.client = client
        self.vector_store = vector_store
        self.relevance_checker = relevance_checker
        self.model_name = model_name

    def generate(self, query: str, memory_context: str = '') -> dict:
        results = self.vector_store.search(query, top_k=3)
        retrieved_context = "\n\n".join([res['content'] for res in results])
        
        is_relevant = False
        if retrieved_context:
            is_relevant = self.relevance_checker.check(query, retrieved_context)
            
        if not is_relevant:
            return {
                'code': None,
                'explanation': None,
                'needs_feedback': True,
                'message': "I don't have enough information in my knowledge base to answer this query. Could you please provide a solution or more context so I can learn from it?"
            }
            
        messages = [
            {'role': 'system', 'content': 'You are an AI coding assistant. Generate Python code to answer the user query. Use the provided context if helpful. Provide the code in a markdown block, and optionally a brief explanation.'},
            {'role': 'user', 'content': f'Conversation Context:\n{memory_context}\n\nKnowledge Base Context:\n{retrieved_context}\n\nQuery: {query}'}
        ]
        
        response = self.client.chat_completion(
            model=self.model_name,
            messages=messages,
            max_tokens=1500
        )
        content = response.choices[0].message.content
        
        code_match = re.search(r'```python\n(.*?)\n```', content, re.DOTALL)
        if not code_match:
            code_match = re.search(r'```\n(.*?)\n```', content, re.DOTALL)
            
        code_block = code_match.group(1) if code_match else None
        
        return {
            'code': code_block,
            'explanation': content,
            'needs_feedback': False,
            'message': content
        }

# Test RAG Code Generator
generator = RAGCodeGenerator(client, vs, checker, MODEL_NAME)
result = generator.generate("Write a binary search function")
print(f"Needs feedback: {result['needs_feedback']}")
print(f"Message: {result['message'][:200]}...")

## 8. Human Feedback Learning

When the assistant can't find relevant knowledge, the user can provide a solution. The assistant automatically embeds it and stores it in ChromaDB for future use.

In [ ]:
class FeedbackLearner:
    """Learns from user-provided solutions by storing them in the vector database."""
    
    def __init__(self, vector_store: VectorStore):
        self.vector_store = vector_store
        
    def learn(self, user_solution: str, original_query: str = '') -> str:
        content = f"Query: {original_query}\nSolution:\n{user_solution}"
        self.vector_store.add_document(content, {"source": "user_feedback", "query": original_query})
        return "Thank you! I have added this solution to my knowledge base."

# Test Feedback Learning
learner = FeedbackLearner(vs)

# Simulate user providing a solution
result = learner.learn(
    user_solution="""def binary_search(arr, target):\n    left, right = 0, len(arr) - 1\n    while left <= right:\n        mid = (left + right) // 2\n        if arr[mid] == target:\n            return mid\n        elif arr[mid] < target:\n            left = mid + 1\n        else:\n            right = mid - 1\n    return -1""",
    original_query="Write a binary search function"
)
print(result)
print(f"Knowledge base now has {vs.get_collection_count()} documents")

## 9. Conversation Memory

Maintains conversational context across interactions, storing previous questions, generated code, and user preferences.

In [ ]:
class ConversationMemory:
    """Manages conversation history for context-aware interactions."""
    
    def __init__(self, max_turns: int = 10):
        self.max_turns = max_turns
        self.messages = []
        
    def add_message(self, role: str, content: str):
        self.messages.append({'role': role, 'content': content})
        if len(self.messages) > self.max_turns * 2:
            self.messages = self.messages[-(self.max_turns * 2):]
            
    def get_context(self) -> str:
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.messages])
        
    def clear(self):
        self.messages = []
        
    def get_messages(self) -> list:
        return self.messages

# Test Memory
memory = ConversationMemory(max_turns=5)
memory.add_message('user', 'How do I sort a list?')
memory.add_message('assistant', 'You can use the sorted() function.')
memory.add_message('user', 'Can you show me an example?')
print(memory.get_context())

## 10. Code Execution Tool

The code executor runs generated code in a controlled subprocess environment, capturing stdout and stderr with a timeout for safety.

In [ ]:
class CodeExecutor:
    """Executes code in a sandboxed subprocess."""
    
    def execute(self, code: str, timeout: int = 30) -> dict:
        try:
            with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
                f.write(code)
                temp_path = f.name
                
            result = subprocess.run(
                ['python', temp_path],
                capture_output=True,
                text=True,
                timeout=timeout
            )
            
            return {
                'stdout': result.stdout,
                'stderr': result.stderr,
                'success': result.returncode == 0,
                'error': None if result.returncode == 0 else "Process exited with non-zero status"
            }
        except subprocess.TimeoutExpired:
            return {'stdout': '', 'stderr': 'Execution timed out', 'success': False, 'error': 'Timeout'}
        except Exception as e:
            return {'stdout': '', 'stderr': str(e), 'success': False, 'error': str(e)}
        finally:
            if 'temp_path' in locals() and os.path.exists(temp_path):
                try:
                    os.remove(temp_path)
                except:
                    pass

# Test Code Execution
executor = CodeExecutor()
result = executor.execute('print("Hello from Cellula!")')
print(f"Success: {result['success']}")
print(f"Output: {result['stdout']}")

## 11. Main Orchestrator — CodingAssistant

The `CodingAssistant` class ties everything together. It:
1. Classifies user intent
2. Routes to the correct pipeline
3. Manages memory
4. Handles feedback learning
5. Executes code

In [ ]:
class CodingAssistant:
    """Main orchestrator that ties all components together."""
    
    def __init__(self, hf_token: str, model_name: str = 'mistralai/Mistral-7B-Instruct-v0.3'):
        self.hf_token = hf_token
        self.model_name = model_name
        self.client = InferenceClient(token=hf_token)
        
        self.vector_store = VectorStore(persist_directory='./chroma_db')
        self.classifier = IntentClassifier(self.client, self.model_name)
        self.explainer = CodeExplainer(self.client, self.model_name)
        self.relevance_checker = RelevanceChecker(self.client, self.model_name)
        self.generator = RAGCodeGenerator(
            self.client, self.vector_store, self.relevance_checker, self.model_name
        )
        self.learner = FeedbackLearner(self.vector_store)
        self.memory = ConversationMemory()
        self.executor = CodeExecutor()

    def process_query(self, query: str) -> dict:
        """Process a user query through the full pipeline."""
        self.memory.add_message('user', query)
        memory_context = self.memory.get_context()
        
        intent = self.classifier.classify(query)
        print(f"[Intent: {intent}]")
        
        if intent == 'EXPLAIN':
            explanation = self.explainer.explain(query, memory_context)
            self.memory.add_message('assistant', explanation)
            return {
                'type': 'explain',
                'response': explanation,
                'code': None,
                'needs_feedback': False
            }
        else:
            result = self.generator.generate(query, memory_context)
            self.memory.add_message('assistant', result['message'])
            return {
                'type': 'generate',
                'response': result['message'],
                'code': result['code'],
                'needs_feedback': result['needs_feedback']
            }

    def provide_feedback(self, solution: str, original_query: str = '') -> str:
        """Learn from user-provided feedback."""
        msg = self.learner.learn(solution, original_query)
        self.memory.add_message('user', f"Feedback provided: {solution}")
        self.memory.add_message('assistant', msg)
        return msg

    def execute_code(self, code: str) -> dict:
        """Execute generated code."""
        return self.executor.execute(code)

## 12. Test the Full Pipeline

Let's test the complete assistant with various queries:

In [ ]:
# Initialize the assistant
assistant = CodingAssistant(hf_token=HF_TOKEN, model_name=MODEL_NAME)

# Test 1: Code Explanation (Route 1 — no RAG)
print("="*60)
print("TEST 1: Code Explanation")
print("="*60)
result = assistant.process_query("""Explain this code:\ndef fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)""")
print(result['response'])

In [ ]:
# Test 2: Code Generation (Route 2 — with RAG)
print("="*60)
print("TEST 2: Code Generation")
print("="*60)
result = assistant.process_query("Write a binary search function")
print(f"Needs feedback: {result['needs_feedback']}")
print(result['response'])

# If code was generated, execute it
if result['code']:
    print("\n" + "="*60)
    print("CODE EXECUTION RESULT")
    print("="*60)
    exec_result = assistant.execute_code(result['code'])
    print(f"Success: {exec_result['success']}")
    print(f"Output: {exec_result['stdout']}")
    if exec_result['stderr']:
        print(f"Errors: {exec_result['stderr']}")

In [ ]:
# Test 3: Feedback Learning
print("="*60)
print("TEST 3: Feedback Learning")
print("="*60)

# Provide a solution the assistant can learn
feedback = assistant.provide_feedback(
    user_solution="""def merge_sort(arr):\n    if len(arr) <= 1:\n        return arr\n    mid = len(arr) // 2\n    left = merge_sort(arr[:mid])\n    right = merge_sort(arr[mid:])\n    return merge(left, right)\n\ndef merge(left, right):\n    result = []\n    i = j = 0\n    while i < len(left) and j < len(right):\n        if left[i] <= right[j]:\n            result.append(left[i])\n            i += 1\n        else:\n            result.append(right[j])\n            j += 1\n    result.extend(left[i:])\n    result.extend(right[j:])\n    return result""",
    original_query="Implement merge sort"
)
print(feedback)
print(f"Knowledge base size: {assistant.vector_store.get_collection_count()} documents")

# Now try asking about merge sort — should find relevant context
result = assistant.process_query("Implement merge sort")
print(f"\nNeeds feedback: {result['needs_feedback']}")
print(result['response'][:300])

## 13. Streamlit App

The Streamlit UI provides a chat interface. To run the app:

```bash
streamlit run app.py
```

The app includes:
- 💬 Chat interface with conversation history
- 🧠 Memory visualization in the sidebar
- 📚 Knowledge base stats
- ▶️ Code execution buttons
- 📁 File upload support
- 🔄 Human feedback learning loop

In [ ]:
# Display the Streamlit app code for reference
print("To launch the Streamlit UI, run the following in your terminal:")
print("  streamlit run app.py")
print(f"\nMake sure to set your HF token in the sidebar.")
print(f"Current knowledge base: {assistant.vector_store.get_collection_count()} documents")